In [1]:
import numpy as np
import pandas as pd
import random

In [2]:
FOLD = 0

In [3]:
df = pd.read_csv(f"/scratch1/smaruj/genomic_flat_regions/flat_regions_chrom_states_tsv/fold{FOLD}_selected_genomic_windows_centered_chrom_states.tsv", sep="\t")

In [ ]:
df

In [4]:
def one_hot_encode_sequence(sequence_obj):
    # Convert pyfaidx.Sequence object to string
    sequence = str(sequence_obj).upper()
    
    # Define the mapping from bases to integers
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    valid_bases = list(base_to_int.keys())

    # Step 1: Convert sequence to integer encoding with random base for 'N'
    encoded_indices = []
    for base in sequence:
        if base in base_to_int:
            encoded_indices.append(base_to_int[base])
        else:
            random_base = random.choice(valid_bases)
            encoded_indices.append(base_to_int[random_base])

    # Step 2: One-hot encode the sequence
    encoded_sequence = np.array(encoded_indices)
    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    return one_hot_encoded

In [5]:
from torch.utils.data import Dataset, DataLoader
import torch

In [6]:
df.columns

Index(['chrom', 'fold', 'PearsonR', 'centered_start', 'centered_end',
       'centered_flat_start', 'centered_flat_end', 'active_fraction',
       'neutral_fraction', 'repressive_fraction'],
      dtype='object')

In [ ]:
class GenomicSequenceDataset(Dataset):
    def __init__(self, coord_df, genome_fasta):
        self.coords = coord_df
        self.genome = genome_fasta

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        TARGET_LEN = 1310720
        row = self.coords.iloc[idx]
        chrom = row["chrom"]
        start = row["centered_start"]
        end = row["centered_end"]

        seq = self.genome[chrom][start:end].seq.upper()
        if len(seq) != TARGET_LEN:
            seq = seq[:TARGET_LEN].ljust(TARGET_LEN, 'N')

        one_hot = one_hot_encode_sequence(seq)
        return torch.from_numpy(one_hot.copy())

In [ ]:
from pyfaidx import Fasta

In [ ]:
fasta_file = "/project2/fudenber_735/genomes/mm10/mm10.fa"
genome = Fasta(fasta_file)

In [ ]:
orig_dataset = GenomicSequenceDataset(df, genome)

orig_loader = DataLoader(orig_dataset, batch_size=16, shuffle=False)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

from akita_model.model import SeqNN

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/models/finetuned/mouse/Hsieh2019_mESC/checkpoints/Akita_v2_mouse_Hsieh2019_mESC_model0_finetuned.pth", map_location=device))
model.eval()

In [ ]:
preds_all = []

model.eval()
with torch.no_grad():
    for batch in orig_loader:
        batch = batch.to(device)  # shape [B, 4, L]
        
        # Predict
        preds = model(batch).cpu()  # shape [B, T]
        preds_all.extend(preds)

In [ ]:
preds_all = torch.cat(preds_all, dim=0)

In [ ]:
# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value

def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
df

In [ ]:
for i in range(10):
    print(i)
    matrix = from_upper_triu(preds_all[i,:], matrix_len=512, num_diags=2)
    
    flat_start = df.loc[i, "centered_flat_start"]
    flat_end = df.loc[i, "centered_flat_end"]
    
    plt.figure(figsize=(5, 5))
    ax = plt.gca()  # Get the current axes
    
    im = ax.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.colorbar(im)

    # Add red square along diagonal if flat_start/end are valid
    if not np.isnan(flat_start) and not np.isnan(flat_end):
        width = flat_end - flat_start
        rect = patches.Rectangle(
            (flat_start, flat_start),  # (x, y) = bottom-left corner
            width,                     # width and height are same for diagonal square
            width,
            linewidth=1.5,
            edgecolor='red',
            facecolor='none'
        )
        ax.add_patch(rect)

    plt.title(f"Example {i}")
    plt.show()